In [ ]:
# | default_exp visualization

# Visualization Tool

> Defining tools for model visualization

In [ ]:
# | export
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.animation import FuncAnimation
from torch import nn

In [ ]:
# | export
def plot_decision_boundary_movie(
    model: nn.Module,  # PyTorch model.
    state_dicts: list[
        nn.Module.state_dict
    ],  # List of state dicts to build decision boundary
    X: torch.Tensor,  # Input data
    y: torch.Tensor,  # Label vector
    figsize: tuple[int] = (8, 8),  # Size of figure
    labels: list[int] = [0, 1],  # Labels
):
    # Initialization
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
    x_in = np.c_[xx.ravel(), yy.ravel()]
    x_in = torch.tensor(x_in, dtype=torch.float32).to(
        next(model.parameters()).device
    )  # Move to GPU

    # Figure and plot
    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(
        X[y == labels[0], 0],
        X[y == labels[0], 1],
        c="crimson",
        marker="o",
        s=50,
        label="Class -1",
    )
    ax.scatter(
        X[y == labels[1], 0],
        X[y == labels[1], 1],
        c="royalblue",
        marker="o",
        s=50,
        label="Class 1",
    )

    # Initialize `contour` outside of the update function
    contour = ax.contourf(
        xx,
        yy,
        np.zeros_like(xx),
        cmap="coolwarm_r",
        alpha=0.3,
        levels=np.linspace(labels[0], labels[1], 3),
    )

    # Precompute predictions for all state_dicts
    y_preds = []
    for model_state_dict in state_dicts:
        model.load_state_dict(model_state_dict)
        model.eval()
        out = model(x_in)
        out = torch.argmax(out, dim=1)
        with torch.no_grad():
            y_preds.append(out.cpu().numpy().reshape(xx.shape))

    # Update function with precomputed predictions
    def update(frame):
        nonlocal contour  # Use the outer scope's contour variable

        y_pred = y_preds[frame]

        for coll in contour.collections:
            coll.remove()
        contour = ax.contourf(
            xx,
            yy,
            y_pred,
            cmap="coolwarm_r",
            alpha=0.3,
            levels=np.linspace(labels[0], labels[1], 3),
        )
        return contour.collections

    # Animation
    ani = FuncAnimation(fig, update, frames=len(state_dicts), blit=True, repeat=False)
    return ani